# ONNX -> TensorRT (FP16 / INT8) on a T4 vs the CPU numbers

> **Not executed in this repo's CPU container.** Written to run on Colab (Runtime -> GPU). No numbers in this notebook were produced by the author; fill the results table from your run and record hardware/date in `results/RESUME_METRICS.md`.

Compares per-part latency/throughput and cost per 1,000 parts of the PatchCore feature extractor on a Colab T4 (ONNX Runtime CUDA and TensorRT execution providers) with the CPU results in `results/edge_summary.md`. FAISS kNN search stays on CPU here (as in the CPU benchmark); say so when you quote numbers.

In [ ]:
!git clone -q https://github.com/karthik-vi/edge-defect-inspection.git repo
%cd repo
!pip -q install onnxruntime-gpu onnx tensorrt torch torchvision faiss-cpu opencv-python-headless scikit-learn pyyaml pydantic-settings
!pip -q install -e . --no-deps
!bash scripts/download_data.sh   # 3 MVTec AD categories only (CC BY-NC-SA 4.0, non-commercial)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python scripts/export_onnx.py 32            # models/r18_fp32.onnx and models/r18_int8.onnx (QDQ)

In [ ]:
import time, numpy as np, onnxruntime as ort
print(ort.get_available_providers())
def session(path, kind):
    opts = {"fp32_cuda": [("CUDAExecutionProvider", {})],
            "trt_fp16": [("TensorrtExecutionProvider", {"trt_fp16_enable": True, "trt_engine_cache_enable": True,
                                                        "trt_engine_cache_path": "trt_cache_fp16"})],
            "trt_int8": [("TensorrtExecutionProvider", {"trt_int8_enable": True, "trt_fp16_enable": True,
                                                        "trt_engine_cache_enable": True, "trt_engine_cache_path": "trt_cache_int8"})]}[kind]
    return ort.InferenceSession(path, providers=opts + [("CPUExecutionProvider", {})])

def bench(sess, bs, n=200, warm=20):
    x = np.random.rand(bs, 3, 224, 224).astype(np.float32); ms = []
    for i in range(n + warm):
        t = time.perf_counter(); sess.run(None, {"image": x}); ms.append((time.perf_counter() - t) * 1e3)
    ms = np.array(ms[warm:]); return {"bs": bs, "p50": np.percentile(ms, 50), "p95": np.percentile(ms, 95), "p99": np.percentile(ms, 99),
                                     "parts_per_s": bs * 1e3 / np.percentile(ms, 50)}
rows = []
for name, path in [("fp32_cuda", "models/r18_fp32.onnx"), ("trt_fp16", "models/r18_fp32.onnx"), ("trt_int8", "models/r18_int8.onnx")]:
    s = session(path, name)
    for bs in (1, 8): rows.append({"backend": name, **bench(s, bs)})
import pandas as pd; df = pd.DataFrame(rows).round(2); df

In [ ]:
# Cost per 1,000 parts. PRICE IS YOUR INPUT (instance $/hour for the T4 you used); it is not a measurement.
T4_USD_PER_HOUR = None   # e.g. your cloud's on-demand T4 price
if T4_USD_PER_HOUR:
    df["usd_per_1k_parts"] = 1000 / df.parts_per_s / 3600 * T4_USD_PER_HOUR
df

In [ ]:
# Accuracy check for the INT8 engine: rerun the repo's evaluation with this session as the extractor
# (see scripts/bench_edge.py, OrtExt). Record image/pixel AUROC next to the CPU INT8 numbers.

## Results (fill in after running)
| backend | batch | p50 / p99 ms | parts/s | $/1k parts | image AUROC |
|---|---|---|---|---|---|
| CPU ORT INT8, 4 threads (from `results/edge_summary.md`) | 1 | 25 / 37 | 40 | | 0.968 |
| T4 CUDA FP32 | 1 | | | | |
| T4 TensorRT FP16 | 1 | | | | |
| T4 TensorRT INT8 | 1 | | | | |

Decision to record in `docs/DECISIONS.md`: does a GPU pay off at the line's actual rate (parts/s)? A T4 is idle most of the time at one part every few seconds; the CPU result may win on cost.